## Setup

Run the next cell first on Google Colab or Kaggle (internet access required). It installs XWhy from GitHub; you can skip it if XWhy is already installed locally.

In [ ]:
# Setup for Google Colab / Kaggle: install the latest XWhy from GitHub.
# Skip this cell if XWhy is already installed in your environment.
%pip install -q "xwhy @ git+https://github.com/Dependable-Intelligent-Systems-Lab/xwhy.git"

from importlib.metadata import version
print("xwhy", version("xwhy"))

# Part 1: Download Dataset and Model Training Code

In [ ]:
!pip install -q torch-geometric

In [ ]:
import os
import warnings
import math, random
from pathlib import Path
from typing import Any, Dict

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch_geometric.io import read_off
import torch.nn as nn
import torch.nn.functional as F

import xwhy
from xwhy.plots import (
    display_plotly_figure,
    plot_3d_mesh,
    plot_3d_point_cloud,
    plot_colored_3d_point_cloud,
    plot_point_cloud_clusters,
    plot_point_cloud,
)

warnings.filterwarnings('ignore')

random.seed(42)
TRAIN_MODEL = False
DOWNLOAD_DATASET = False

import os.path as osp
from torch_geometric.data import download_url, extract_zip

root = 'data/ModelNet40'
url = 'http://modelnet.cs.princeton.edu/ModelNet40.zip'
raw_dir = osp.join(root, 'raw')

if DOWNLOAD_DATASET:
    os.makedirs(root, exist_ok=True)
    print("Downloading ModelNet40 dataset...")
    path_zip = download_url(url, root)
    extract_zip(path_zip, root)
    os.unlink(path_zip)
    folder = osp.join(root, 'ModelNet40')
    if osp.exists(raw_dir):
        import shutil
        shutil.rmtree(raw_dir)
    os.rename(folder, raw_dir)
    print("Dataset download and extraction complete.")

path = Path(raw_dir)

In [ ]:
class PointSampler:
    def __init__(self, output_size: int) -> None:
        self.output_size = output_size

    def triangle_area(self, p1, p2, p3) -> float:
        if not (np.all(np.isfinite(p1)) and np.all(np.isfinite(p2)) and np.all(np.isfinite(p3))):
            return 0.0
        side_a = np.linalg.norm(p1 - p2)
        side_b = np.linalg.norm(p2 - p3)
        side_c = np.linalg.norm(p3 - p1)
        if not (np.isfinite(side_a) and np.isfinite(side_b) and np.isfinite(side_c)) or side_a <= 0 or side_b <= 0 or side_c <= 0:
            return 0.0
        s = 0.5 * (side_a + side_b + side_c)
        arg = s * (s - side_a) * (s - side_b) * (s - side_c)
        return arg ** 0.5 if arg > 0 else 0.0

    def sample_point(self, p1, p2, p3):
        s, t = sorted([random.random(), random.random()])
        return (
            s * p1[0] + (t - s) * p2[0] + (1 - t) * p3[0],
            s * p1[1] + (t - s) * p2[1] + (1 - t) * p3[1],
            s * p1[2] + (t - s) * p2[2] + (1 - t) * p3[2],
        )

    def __call__(self, mesh) -> np.ndarray:
        vertices, faces = mesh
        vertices = np.array(vertices)
        areas = [self.triangle_area(vertices[f[0]], vertices[f[1]], vertices[f[2]]) for f in faces]
        areas = np.nan_to_num(np.array(areas), nan=0.0)
        if np.sum(areas) == 0:
            areas = np.ones_like(areas) if len(areas) > 0 else np.zeros((self.output_size, 3))
        indices = random.choices(range(len(faces)), weights=areas, k=self.output_size)
        return np.array([self.sample_point(*vertices[faces[i]]) for i in indices])

class Normalize:
    def __call__(self, pc: np.ndarray) -> np.ndarray:
        norm = pc - np.mean(pc, axis=0)
        return norm / np.max(np.linalg.norm(norm, axis=1))

class RandomRotationZ:
    def __call__(self, pc: np.ndarray) -> np.ndarray:
        assert len(pc.shape) == 2
        theta = random.random() * 2.0 * math.pi
        rot = np.array([
            [math.cos(theta), -math.sin(theta), 0],
            [math.sin(theta), math.cos(theta), 0],
            [0, 0, 1],
        ])
        return rot.dot(pc.T).T

class RandomNoise:
    def __call__(self, pc: np.ndarray) -> np.ndarray:
        assert len(pc.shape) == 2
        return pc + np.random.normal(0, 0.02, pc.shape)

class ToTensor:
    def __call__(self, pc: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(pc)

def default_transforms() -> transforms.Compose:
    return transforms.Compose([PointSampler(1024), Normalize(), ToTensor()])

def build_train_transforms() -> transforms.Compose:
    return transforms.Compose([
        PointSampler(1024),
        Normalize(),
        RandomRotationZ(),
        RandomNoise(),
        ToTensor(),
    ])

In [ ]:
class PointCloudData(Dataset):
    def __init__(self, root_dir: Path, valid: bool = False, folder: str = "train", transform=None):
        self.root_dir = root_dir
        if self.root_dir.exists():
            folders = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(root_dir / d)]
            self.classes = {c: i for i, c in enumerate(folders)}
        else:
            self.classes = {}
            
        self.transforms = transform if not valid else default_transforms()
        self.files = []
        if self.root_dir.exists():
            for cat in self.classes:
                cat_dir = root_dir / Path(cat) / folder
                if cat_dir.exists():
                    for f in os.listdir(cat_dir):
                        if f.endswith(".off"):
                            self.files.append({"pcd_path": cat_dir / f, "category": cat})

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        info = self.files[index]
        data = read_off(info["pcd_path"])
        pc = self.transforms((data.pos.numpy(), data.face.numpy().T))
        return {"pointcloud": pc, "category": self.classes[info["category"]]}

if TRAIN_MODEL:
    train_transforms = build_train_transforms()
    train_dataset = PointCloudData(path, transform=train_transforms)
    valid_dataset = PointCloudData(path, valid=True, folder="test", transform=default_transforms())

    train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
    valid_loader = DataLoader(dataset=valid_dataset, batch_size=64)

In [ ]:
class TNet(nn.Module):
    def __init__(self, k: int = 3) -> None:
        super().__init__()
        self.k = k
        self.conv1, self.conv2, self.conv3 = nn.Conv1d(k, 64, 1), nn.Conv1d(64, 128, 1), nn.Conv1d(128, 1024, 1)
        self.fc1, self.fc2, self.fc3 = nn.Linear(1024, 512), nn.Linear(512, 256), nn.Linear(256, k * k)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(64), nn.BatchNorm1d(128), nn.BatchNorm1d(1024)
        self.bn4, self.bn5 = nn.BatchNorm1d(512), nn.BatchNorm1d(256)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch_size = inputs.size(0)
        x = F.relu(self.bn1(self.conv1(inputs)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = nn.MaxPool1d(x.size(-1))(x)
        x = nn.Flatten(1)(x)
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        identity = torch.eye(self.k, device=inputs.device).repeat(batch_size, 1, 1)
        return self.fc3(x).view(-1, self.k, self.k) + identity

class Transform(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.input_transform, self.feature_transform = TNet(k=3), TNet(k=64)
        self.conv1, self.conv2, self.conv3 = nn.Conv1d(3, 64, 1), nn.Conv1d(64, 128, 1), nn.Conv1d(128, 1024, 1)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(64), nn.BatchNorm1d(128), nn.BatchNorm1d(1024)

    def forward(self, inputs: torch.Tensor):
        m3x3 = self.input_transform(inputs)
        x = torch.bmm(inputs.transpose(1, 2), m3x3).transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        m64x64 = self.feature_transform(x)
        x = torch.bmm(x.transpose(1, 2), m64x64).transpose(1, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        return nn.Flatten(1)(nn.MaxPool1d(x.size(-1))(x)), m3x3, m64x64

class PointNet(nn.Module):
    def __init__(self, num_classes: int = 40) -> None:
        super().__init__()
        self.transform = Transform()
        self.fc1, self.fc2, self.fc3 = nn.Linear(1024, 512), nn.Linear(512, 256), nn.Linear(256, num_classes)
        self.bn1, self.bn2 = nn.BatchNorm1d(512), nn.BatchNorm1d(256)
        self.dropout, self.log_softmax = nn.Dropout(p=0.3), nn.LogSoftmax(dim=1)

    def forward(self, inputs: torch.Tensor):
        x, m3x3, m64x64 = self.transform(inputs)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.dropout(self.fc2(x))))
        return self.log_softmax(self.fc3(x)), m3x3, m64x64

def pointnet_loss(outputs, labels, m3x3, m64x64, alpha=1e-4):
    criterion = nn.NLLLoss()
    bs = outputs.size(0)
    id3 = torch.eye(3, device=outputs.device).repeat(bs, 1, 1)
    id64 = torch.eye(64, device=outputs.device).repeat(bs, 1, 1)
    diff3 = id3 - torch.bmm(m3x3, m3x3.transpose(1, 2))
    diff64 = id64 - torch.bmm(m64x64, m64x64.transpose(1, 2))
    return criterion(outputs, labels) + alpha * (torch.norm(diff3) + torch.norm(diff64)) / float(bs)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0008)

if TRAIN_MODEL:
    def train(model, train_loader, val_loader, epochs=3):
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            for batch_idx, batch in enumerate(train_loader):
                inputs = batch["pointcloud"].to(device).float()
                labels = batch["category"].to(device)
                optimizer.zero_grad()
                outputs, m3x3, m64x64 = model(inputs.transpose(1, 2))
                loss = pointnet_loss(outputs, labels, m3x3, m64x64)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                if batch_idx % 5 == 4:
                    print(f"[Epoch: {epoch+1}, Batch: {batch_idx+1}/{len(train_loader)}] loss: {running_loss / 5:.3f}")
                    running_loss = 0.0
            
            # Validation
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for batch in val_loader:
                    inputs = batch["pointcloud"].to(device).float()
                    labels = batch["category"].to(device)
                    outputs, _, _ = model(inputs.transpose(1, 2))
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            print(f"Validation accuracy: {100.0 * correct / total:.2f}%")
            torch.save(model.state_dict(), f"save-{epoch + 1}.pth")

    train(model, train_loader, valid_loader, epochs=3)
else:
    import gdown
    os.makedirs("data", exist_ok=True)
    if not osp.exists("data/save.pth"):
        gdown.download(id="1zLl0E9akEOneDGJSkFATdtox9COpbd1F", output="data/save.pth", quiet=False)
    if not osp.exists("data/all_data.pt"):
        gdown.download(id="1Kv98MSvAzx20QLoy8o9q14Tx7n681dn3", output="data/all_data.pt", quiet=False)
        
    model.load_state_dict(torch.load("data/save.pth", map_location=device))

model.eval()

# Load evaluation data tensor/labels
loaded = torch.load("data/all_data.pt", map_location="cpu")
all_inputs, all_labels = loaded["all_inputs"], loaded["all_labels"]

# Part 2: Standard Point Cloud Explainer & Stability Analysis

In [ ]:
from xwhy.explainers import PointCloudExplainer
from xwhy.surrogate import SurrogateType
from xwhy.metrics.point_cloud import (
    compute_noisy_explanations,
    calculate_jaccard_stability_score,
)
from xwhy.plots import plot_point_cloud

# Configuration
random_seed = 42
num_clusters = 32
specified_object_index = 1
num_perturbations = 500
removal_probability = 0.5
num_iterations = 10
num_new_points = 30
sphere_radius = 0.07

sample_input = all_inputs[specified_object_index]
sample_label = all_labels[specified_object_index]

In [ ]:
# ==============================================================================
# Standard PyTorch Model Evaluation (KMeans Mode)
# ==============================================================================
explainer_case1 = PointCloudExplainer(
    model=model,
    num_clusters=num_clusters,
    num_perturbations=num_perturbations,
    removal_probability=removal_probability,
    clustering_mode="kmeans",
)

result_case1 = explainer_case1.explain(
    instance=sample_input,
    sample_label=sample_label,
    # cluster_labels=None is implicit; kmeans handles it natively
)
print("Standard Explanation Metrics:", result_case1.metrics)

# Saves to an interactive HTML file without attempting to display in console
fig = plot_point_cloud(
    result_case1, 
    save_path="output.html", 
    show=False,
    base_color="gray", 
    highlight_color="orange"
)

important_clusters_case1 = compute_noisy_explanations(
    sample_input=sample_input,
    sample_label=sample_label,
    model=model,
    cluster_labels=None,
    clustering_mode="kmeans",
    num_clusters=num_clusters,
    num_perturbations=num_perturbations,
    removal_probability=removal_probability,
    num_iterations=num_iterations,
    num_new_points=num_new_points,
    sphere_radius=sphere_radius,
    seed=random_seed,
    surrogate_type=SurrogateType.LIME,
)

_, mean_jaccard_case1 = calculate_jaccard_stability_score(important_clusters_case1)
print(f"Case 1 Mean Jaccard Similarity: {mean_jaccard_case1:.4f}")

In [ ]:
display_plotly_figure(fig)

# Part 3: Hugging Face PointNet using CustomPointCloudModel

In [ ]:
import importlib.util
import sys
import types

import torch
from huggingface_hub import hf_hub_download

from xwhy.models.point_cloud.custom import CustomPointCloudModel

# ------------------------------------------------------------------------------
# 1. Download the Hugging Face model source and weights
# ------------------------------------------------------------------------------
hf_repo_id = "DavidHanSZ/pointnet-modelnet40"

hf_source_path = hf_hub_download(
    repo_id=hf_repo_id,
    filename="pointnet_modelnet40.py",
)

hf_weights_path = hf_hub_download(
    repo_id=hf_repo_id,
    filename="pytorch_model.bin",
)

# ------------------------------------------------------------------------------
# 2. Load PointNetClassification from the Hugging Face repository
# ------------------------------------------------------------------------------
if "trackio" not in sys.modules:
    sys.modules["trackio"] = types.ModuleType("trackio")

if "datasets" not in sys.modules:
    datasets_module = types.ModuleType("datasets")
    datasets_module.load_dataset = None
    sys.modules["datasets"] = datasets_module

spec = importlib.util.spec_from_file_location(
    "hf_pointnet_modelnet40",
    hf_source_path,
)

if spec is None or spec.loader is None:
    raise ImportError(
        f"Could not load Hugging Face model source: {hf_source_path}"
    )

hf_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = hf_module
spec.loader.exec_module(hf_module)

PointNetClassification = hf_module.PointNetClassification

# ------------------------------------------------------------------------------
# 3. Instantiate the Hugging Face model and load pretrained weights
# ------------------------------------------------------------------------------
hf_model = PointNetClassification(
    num_classes=40,
    dropout=0.3,
)

state_dict = torch.load(
    hf_weights_path,
    map_location=device,
)

hf_model.load_state_dict(state_dict)
hf_model = hf_model.to(device)
hf_model.eval()

# ------------------------------------------------------------------------------
# 4. Map the Hugging Face model into CustomPointCloudModel
# ------------------------------------------------------------------------------
hf_custom_model = CustomPointCloudModel(
    model=hf_model,
)

In [ ]:
# ------------------------------------------------------------------------------
# 5. Create the Point Cloud Explainer
# ------------------------------------------------------------------------------
explainer_case2 = PointCloudExplainer(
    model=hf_custom_model,
    num_clusters=num_clusters,
    num_perturbations=num_perturbations,
    removal_probability=removal_probability,
    clustering_mode="kmeans",
)

# ------------------------------------------------------------------------------
# 6. Generate explanation
# ------------------------------------------------------------------------------
result_case2 = explainer_case2.explain(
    instance=sample_input,
    sample_label=sample_label,
)

print("Hugging Face Explanation Metrics:", result_case2.metrics)

# ------------------------------------------------------------------------------
# 7. Plot explanation
# ------------------------------------------------------------------------------
fig = plot_point_cloud(
    result_case2,
    save_path="output_huggingface.html",
    show=False,
    base_color="gray",
    highlight_color="orange",
)

# ------------------------------------------------------------------------------
# 8. Stability analysis
# ------------------------------------------------------------------------------
important_clusters_case2 = compute_noisy_explanations(
    sample_input=sample_input,
    sample_label=sample_label,
    model=hf_custom_model,
    cluster_labels=None,
    clustering_mode="kmeans",
    num_clusters=num_clusters,
    num_perturbations=num_perturbations,
    removal_probability=removal_probability,
    num_iterations=num_iterations,
    num_new_points=num_new_points,
    sphere_radius=sphere_radius,
    seed=random_seed,
    surrogate_type=SurrogateType.LIME,
)

_, mean_jaccard_case2 = calculate_jaccard_stability_score(
    important_clusters_case2
)

print(
    f"Case 2 Mean Jaccard Similarity: "
    f"{mean_jaccard_case2:.4f}"
)

In [ ]:
display_plotly_figure(fig)

In [ ]:
result = result_case1

# Visualizations

In [ ]:
xwhy.plots.bar(result)

In [ ]:
xwhy.plots.waterfall(result)

In [ ]:
xwhy.plots.force(result)

In [ ]:
xwhy.plots.decision(result)

In [ ]:
xwhy.plots.plot_feature_bar_chart(result)

In [ ]:
xwhy.plots.plot_feature_box_plot(result)